# Treino e backtest do modelo DemandA — admissões t+1 por UF × subgrupo

Consome `workspace.gold.ml_features_mensal_uf_cbo` (notebook 035, versão vigente) e compara modelos para prever as admissões do mês seguinte. **Janela: 2022-01 a 2025-12** — as features INEP ficam nulas a partir de 2026-01 (o Censo 2025 ainda não foi ingerido), descoberta registrada na exploração do 035; como o screening OLS as descartou (p>0,25), o modelo final usa apenas features CAGED defasadas + calendário, e previsões de 2026 continuam possíveis.

## Enquadramento

* **Série**: mensal, regular (painel denso do 035, 698 séries = UF × subgrupo × 68 meses)
* **Chaves**: `sigla_uf` × `codigo_cbo_subgrupo` · **Alvo**: `admissoes` no mês t · **Horizonte**: h = 1 (próximo mês)
* **Covariadas**: todas com lag ≥ 1 mês — nenhuma variável do próprio mês t entra no modelo (sem vazamento, regra de ouro do projeto)
* **Validação**: backtest em janela expansiva, 12 folds mensais (cutoffs 2025-01 a 2025-12): cada fold treina em tudo < cutoff e prevê o mês do cutoff. Nunca embaralha.
* **Baselines obrigatórias**: naive último valor (lag 1) e naive sazonal (lag 12 — repetir a última "estação"). Se o modelo não vence a naive sazonal, a naive é preferida.
* **Modelos**: com 698 séries, a prática recomendada é um regressor **global** (LightGBM com lags/rollings + identificador da série como categórica), não ARIMA/ETS por série. O **SARIMAX por série** roda como referência clássica apenas nas TOP-20 séries por volume (limitação documentada). OLS pooled replica o screening como baseline linear.
* **Métricas**: **MASE** (primária, comparável entre séries), MAE, RMSE, sMAPE. MAPE evitado — o alvo tem zeros (painel denso).
* **MLflow**: runs de backtest por modelo + modelo final (LightGBM refitado em toda a janela) logado como pyfunc que recebe **linhas futuras** (chaves + competência) e devolve a previsão.
* **Gravação idempotente**: `workspace.gold.ml_backtest_admissoes`, append por `versao_backtest` (sem DROP/overwrite).

In [0]:
%pip install statsmodels lightgbm
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import numpy as np
import pandas as pd

dbutils.widgets.text("tabela_features", "workspace.gold.ml_features_mensal_uf_cbo", "Tabela de features (wide table do 035)")
dbutils.widgets.text("versao_transformacao", "0.1.0", "Versão do snapshot de features")
dbutils.widgets.text("tabela_backtest", "workspace.gold.ml_backtest_admissoes", "Tabela-alvo do backtest")
dbutils.widgets.text("versao_backtest", "0.1.0", "Versão deste backtest")
dbutils.widgets.text("experimento_mlflow", "/Users/kevyn.zap@hotmail.com/demandA_admissoes_h1", "Experimento MLflow (caminho absoluto no workspace)")
dbutils.widgets.text("top_sarima", "20", "Nº de séries (top por volume) para o SARIMAX de referência")

TABELA_FEATURES = dbutils.widgets.get("tabela_features")
VERSAO = dbutils.widgets.get("versao_transformacao")
TABELA_BACKTEST = dbutils.widgets.get("tabela_backtest")
VERSAO_BACKTEST = dbutils.widgets.get("versao_backtest")
EXPERIMENTO = dbutils.widgets.get("experimento_mlflow")
TOP_SARIMA = int(dbutils.widgets.get("top_sarima"))

# Janela: 2022-01 (lag_12 disponível) a 2025-12 (último mês com features INEP completas).
# O mês 2026-01 é carregado EXCLUSIVAMENTE para reconstruir o histórico de salário/desligamento
# necessário ao modelo logado (linhas futuras) — não entra no treino.
JANELA_INICIO = "2022-01-01"
JANELA_FIM = "2025-12-31"
HIST_EXTRA = "2026-01-31"
FOLDS = pd.date_range("2025-01-01", "2025-12-01", freq="MS")  # 12 folds, janela expansiva
HORIZONTE = 1
DT_PROCESSAMENTO = datetime.now()

# Chaves e alvo
KEYS = ["sigla_uf", "codigo_cbo_subgrupo"]
ALVO = "admissoes"

print(f"Features: {TABELA_FEATURES} (v{VERSAO}) | Backtest: {TABELA_BACKTEST} (v{VERSAO_BACKTEST})")
print(f"Janela de treino: {JANELA_INICIO} a {JANELA_FIM} | {len(FOLDS)} folds expansivos | horizonte h={HORIZONTE}")
print(f"MLflow: {EXPERIMENTO} | SARIMAX de referência nas top {TOP_SARIMA} séries")

In [0]:
# Carga do painel (leitura única da wide table → pandas para o treino local)
# ~34 mil linhas — pequeno; a construção distribuída permaneceu 100% em Spark no 035.
df_spark = (
    spark.table(TABELA_FEATURES)
    .where(F.col("versao_transformacao") == VERSAO)
    .where((F.col("competencia_mov") >= F.lit(JANELA_INICIO)) & (F.col("competencia_mov") <= F.lit(HIST_EXTRA)))
)
painel = df_spark.toPandas()
painel["competencia_mov"] = pd.to_datetime(painel["competencia_mov"])
painel = painel.sort_values(KEYS + ["competencia_mov"]).reset_index(drop=True)

# Painel de TREINO/BACKTEST: até 2025-12; linha extra de 2026-01 guardada à parte (histórico do wrapper)
hist_extra = painel[painel["competencia_mov"] > JANELA_FIM].copy()
painel = painel[painel["competencia_mov"] <= JANELA_FIM].copy()

# Validações abortáveis do painel
n_series = painel.groupby(KEYS).ngroups
meses_por_serie = painel.groupby(KEYS)["competencia_mov"].nunique()
assert painel[ALVO].isnull().sum() == 0, "Alvo com nulos — abortando"
assert painel[KEYS].groupby(painel["competencia_mov"].dt.to_period("M")).size().min() == n_series, "Painel não é denso — abortando"
share_zeros = (painel[ALVO] == 0).mean()
share_sal_nulo = painel["salario_mediano_lag_1"].isnull().mean()
print(f"Painel: {len(painel)} linhas, {n_series} séries, {painel['competencia_mov'].nunique()} meses ({JANELA_INICIO} a {JANELA_FIM})")
print(f"Regularidade: todas as séries têm {meses_por_serie.iloc[0]} meses")
print(f"Zeros no alvo: {share_zeros:.1%} (painel denso — meses sem contratação são 0 legítimo)")
print(f"Salário mediano nulo (mês anterior sem admissões): {share_sal_nulo:.1%} — tratado como NaN no LightGBM e linha excluída no OLS")
display(painel[["competencia_mov", *KEYS, ALVO]].head(5))

In [0]:
# EDA: séries representativas, sazonalidade e zeros
import matplotlib.pyplot as plt

representativas = [("SP", "212"), ("SC", "231"), ("MG", "214"), ("BA", "223")]
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=False)
for ax, (uf, sub) in zip(axes.flat, representativas):
    s = painel[(painel["sigla_uf"] == uf) & (painel["codigo_cbo_subgrupo"] == sub)]
    ax.plot(s["competencia_mov"], s[ALVO], lw=1)
    ax.set_title(f"{uf} × {sub}", fontsize=10)
    ax.tick_params(labelsize=8)
fig.suptitle("Séries representativas — admissões mensais (2022-01 a 2025-12)", y=1.0)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 3.5))
sazonal = painel.groupby(painel["competencia_mov"].dt.month)[ALVO].mean()
ax.bar(sazonal.index, sazonal.values, color="#4C78A8")
ax.set_title("Sazonalidade — média de admissões por mês do ano (todas as séries)")
ax.set_xlabel("mês")
plt.tight_layout()
plt.show()

print(f"Distribuição do comprimento das séries: todas com {meses_por_serie.max()} meses (>= 2 sazões anuais)")
print(f"Share de zeros por série (mediana): {painel.groupby(KEYS)[ALVO].apply(lambda s: (s == 0).mean()).median():.1%}")

In [0]:
# Features e checagem de vazamento
# Todas as features são lags (>= h=1) do próprio histórico da série: nenhuma variável do mês t
# entra — por construção da wide table (035), não há covariada simultânea. A checagem abaixo
# confirma (correlação alta com o alvo é esperada para LAGS, o que não é vazamento).
painel["mes"] = painel["competencia_mov"].dt.month
for c in ["sigla_uf", "codigo_cbo_subgrupo"]:
    painel[c] = painel[c].astype("category")

# Challenger global (LightGBM) — lags e rollings ricos; árvore tolera colinearidade entre lags
FEAT_LGBM = [
    "admissoes_lag_1", "admissoes_lag_3", "admissoes_lag_12",
    "admissoes_ma_3", "admissoes_ma_12",
    "desligamentos_lag_1", "saldo_lag_1", "salario_mediano_lag_1",
    "mes", "sigla_uf", "codigo_cbo_subgrupo",
]

# Baseline linear (OLS) — UMA medida por conceito, conforme o screening do 035:
# família admissões → ma_12; giro de saída → desl_log_lag_1; preço → salário; calendário → harmônicos
painel["log_admissoes"] = np.log1p(painel[ALVO])
painel["desl_log_lag_1"] = np.log1p(painel["desligamentos_lag_1"].clip(lower=0))
COLS_OLS = ["admissoes_log_ma_12", "desl_log_lag_1", "salario_mediano_lag_1", "mes_sin", "mes_cos"]
FORMULA_OLS = "log_admissoes ~ " + " + ".join(COLS_OLS) + " + C(sigla_uf) + C(mes)"

tabela_tipos = pd.DataFrame({
    "feature": [ALVO] + FEAT_LGBM,
    "tipo": ["numérica inteira (alvo)", "lag 1 do alvo", "lag 3 do alvo", "lag 12 do alvo",
             "rolling (média 3 meses passados)", "rolling (média 12 meses passados)",
             "lag 1 de desligamentos", "lag 1 do saldo adm−desl", "lag 1 do salário mediano",
             "numérica (calendário)", "categórica (id da série)", "categórica (id da série)"],
})
display(tabela_tipos)
corr_alvo = painel[[ALVO] + [c for c in FEAT_LGBM if c not in ("sigla_uf", "codigo_cbo_subgrupo")]].corr()[ALVO].drop(ALVO)
print("Correlação com o alvo (lags — alta por definição, não é vazamento):")
print(corr_alvo.round(3).to_string())

In [0]:
# AUDITORIA PERMANENTE DE VAZAMENTO (data leakage) — gate obrigatório antes do treino
# Recomputa, de forma INDEPENDENTE (funções de janela em SQL), todos os lags e médias móveis
# da wide table e compara com as colunas gravadas. Também confere que as médias móveis NÃO
# incluem o mês corrente e que a feature INEP é constante dentro do ano (dado anual defasado).
# Qualquer violação ABORTA o notebook antes de qualquer treino.
auditoria = spark.sql(f"""
WITH base AS (
  SELECT admissoes_lag_1, admissoes_lag_3, admissoes_lag_12,
         admissoes_ma_3, admissoes_ma_12, desligamentos_lag_1, concluintes_def1,
         admissoes, admissoes_lag_3 AS a_lag3,
         LAG(admissoes, 1)  OVER (PARTITION BY sigla_uf, codigo_cbo_subgrupo ORDER BY competencia_mov) AS chk_lag1,
         LAG(admissoes, 3)  OVER (PARTITION BY sigla_uf, codigo_cbo_subgrupo ORDER BY competencia_mov) AS chk_lag3,
         LAG(admissoes, 12) OVER (PARTITION BY sigla_uf, codigo_cbo_subgrupo ORDER BY competencia_mov) AS chk_lag12,
         LAG(desligamentos, 1) OVER (PARTITION BY sigla_uf, codigo_cbo_subgrupo ORDER BY competencia_mov) AS chk_desl1,
         AVG(admissoes) OVER (PARTITION BY sigla_uf, codigo_cbo_subgrupo ORDER BY competencia_mov ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) AS chk_ma3,
         AVG(admissoes) OVER (PARTITION BY sigla_uf, codigo_cbo_subgrupo ORDER BY competencia_mov ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING) AS chk_ma12,
         AVG(admissoes) OVER (PARTITION BY sigla_uf, codigo_cbo_subgrupo ORDER BY competencia_mov ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ma3_vazada
  FROM {TABELA_FEATURES}
  WHERE versao_transformacao = '{VERSAO}'
)
SELECT
  SUM(CASE WHEN admissoes_lag_1     IS DISTINCT FROM chk_lag1  THEN 1 ELSE 0 END) AS viol_lag1,
  SUM(CASE WHEN admissoes_lag_3     IS DISTINCT FROM chk_lag3  THEN 1 ELSE 0 END) AS viol_lag3,
  SUM(CASE WHEN admissoes_lag_12    IS DISTINCT FROM chk_lag12 THEN 1 ELSE 0 END) AS viol_lag12,
  SUM(CASE WHEN desligamentos_lag_1 IS DISTINCT FROM chk_desl1 THEN 1 ELSE 0 END) AS viol_deslig_lag1,
  SUM(CASE WHEN chk_ma3  IS NOT NULL AND ABS(admissoes_ma_3  - chk_ma3)  > 0.01 THEN 1 ELSE 0 END) AS viol_ma3,
  SUM(CASE WHEN chk_ma12 IS NOT NULL AND ABS(admissoes_ma_12 - chk_ma12) > 0.01 THEN 1 ELSE 0 END) AS viol_ma12,
  -- Média móvel "casando" com a versão que incluiria o mês corrente SEM explicação = vazamento.
  -- Coincidências explicadas (admissões t = admissões t-3, tipicamente séries pequenas com zeros) são legítimas.
  -- Linhas com lag_3 nulo são warm-up (janela parcial) — a janela parcial olha APENAS para trás,
  -- comprovado por viol_ma3 = 0; não é vazamento, apenas menos meses no histórico.
  SUM(CASE WHEN chk_ma3 IS NOT NULL AND ABS(admissoes_ma_3 - ma3_vazada) <= 0.01
                AND admissoes_lag_3 IS NOT NULL AND admissoes != admissoes_lag_3 THEN 1 ELSE 0 END) AS coincidencias_ma3_sem_explicacao,
  SUM(CASE WHEN chk_ma3 IS NOT NULL AND admissoes_lag_3 IS NULL THEN 1 ELSE 0 END) AS warmup_janela_parcial_informativo,
  (SELECT COUNT(*) FROM (
      SELECT sigla_uf, codigo_cbo_subgrupo, year(competencia_mov) AS ano, COUNT(DISTINCT concluintes_def1) AS nd
      FROM {TABELA_FEATURES}
      WHERE versao_transformacao = '{VERSAO}' AND concluintes_def1 IS NOT NULL
      GROUP BY 1, 2, 3 HAVING nd > 1
  )) AS serie_ano_com_def1_variando
FROM base
""").collect()[0]

print(f"Auditoria de vazamento sobre {TABELA_FEATURES} (v{VERSAO}):")
for campo, valor in auditoria.asDict().items():
    print(f"  {campo}: {valor}")

violacoes = {k: v for k, v in auditoria.asDict().items()
            if v > 0 and k != "warmup_janela_parcial_informativo"}
if violacoes:
    raise ValueError(f"AUDITORIA DE VAZAMENTO FALHOU — {violacoes}. Treino abortado.")
print("OK: sem vazamento — todos os lags e médias móveis usam APENAS meses anteriores (janelas SEM o mês corrente);")
print("OK: feature INEP anual defasada é constante dentro de cada ano (fonte Censo defasada);")
print("OK: nenhuma média móvel coincide com a versão 'vazada' fora de coincidências explicadas;")
print(f"OK: warm-up de janela parcial (primeiros meses de cada série, 2021): {auditoria['warmup_janela_parcial_informativo']} linhas — olham apenas para trás e ficam fora da janela de treino (2022+).")

In [0]:
# Métricas: MASE (primária), MAE, RMSE, sMAPE — MAPE evitado (alvo pode ser zero)
EPS = 1e-9

def mae(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = ~(np.isnan(y) | np.isnan(yhat))
    return float(np.mean(np.abs(y[m] - yhat[m])))

def rmse(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = ~(np.isnan(y) | np.isnan(yhat))
    return float(np.sqrt(np.mean((y[m] - yhat[m]) ** 2)))

def smape(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = ~(np.isnan(y) | np.isnan(yhat))
    return float(np.mean(2 * np.abs(y[m] - yhat[m]) / (np.abs(y[m]) + np.abs(yhat[m]) + EPS)))

def mase_denominador(hist_serie):
    """MAE in-sample do naive sazonal (lag 12) — denominação clássica do MASE por série."""
    y = np.asarray(hist_serie, float)
    if len(y) < 13:
        return np.nan
    d = np.abs(y[12:] - y[:-12])
    return float(np.mean(d)) if np.mean(d) > 0 else np.nan  # série constante: MASE indefinido

print("Métricas definidas: MAE, RMSE, sMAPE, MASE (denominador = naive sazonal in-sample por série).")

In [0]:
# Backtest — janela expansiva, 12 folds mensais (cutoff 2025-01 .. 2025-12), h=1, sem shuffle
import warnings
import statsmodels.formula.api as smf
import lightgbm as lgb

registros = []  # previsões linha a linha: modelo, competência, série, real, previsto, denominador MASE

def registrar(modelo, teste, pred):
    r = teste[KEYS + [ALVO]].copy()
    r["modelo"], r["previsto"] = modelo, np.asarray(pred, dtype=float)
    registros.append(r)

for i, cutoff in enumerate(FOLDS):
    treino = painel[painel["competencia_mov"] < cutoff]
    teste = painel[painel["competencia_mov"] == cutoff]

    # Denominador MASE por série (naive sazonal in-sample, calculado no TREINO do fold)
    denom = treino.groupby(KEYS)[ALVO].agg(mase_denominador).rename("mase_den")
    teste_m = teste.merge(denom, left_on=KEYS, right_index=True, how="left")

    # 1) Naive último valor: y_pred = admissões de t-1 (coluna lag_1 da linha de teste)
    registros.append(teste_m.assign(modelo="naive_lag_1", previsto=teste_m["admissoes_lag_1"].astype(float)))
    # 2) Naive sazonal: y_pred = admissões de t-12 (baseline obrigatória)
    registros.append(teste_m.assign(modelo="naive_sazonal_lag_12", previsto=teste_m["admissoes_lag_12"].astype(float)))

    # 3) OLS pooled (baseline linear, mesma fórmula do screening 035) — alvo log1p, previsão expm1
    ols_cols = COLS_OLS + ["log_admissoes", "sigla_uf", "mes"]
    tr_ols = treino.dropna(subset=[c for c in ols_cols if c != "sigla_uf"])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit_ols = smf.ols(FORMULA_OLS, data=tr_ols).fit()
    pred_ols = np.expm1(fit_ols.predict(teste))  # NaN nas features → previsão NaN (excluída da métrica)
    r = teste_m.copy()
    r["modelo"], r["previsto"] = "ols_pooled", pred_ols.values
    registros.append(r)

    # 4) LightGBM global — uma única árvore para as 698 séries (id da série como categórica)
    fit_lgbm = lgb.LGBMRegressor(
        n_estimators=400, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, random_state=42, verbosity=-1,
    ).fit(treino[FEAT_LGBM], treino[ALVO], categorical_feature=["sigla_uf", "codigo_cbo_subgrupo"])
    r = teste_m.copy()
    r["modelo"], r["previsto"] = "lightgbm_global", fit_lgbm.predict(teste[FEAT_LGBM])
    registros.append(r)

    if (i + 1) % 4 == 0:
        print(f"  {len(FOLDS)} folds — concluído até {cutoff.strftime('%Y-%m')} ({(i+1)/len(FOLDS):.0%})")

prev = pd.concat(registros, ignore_index=True)
print(f"\nPrevisões registradas: {len(prev)} linhas ({prev['modelo'].nunique()} modelos × {len(FOLDS)} folds × 698 séries)")
display(prev.groupby("modelo")["previsto"].agg(n="count", nulos=lambda s: s.isna().sum()))

In [0]:
# SARIMAX por série (referência clássica) — apenas nas TOP-N séries por volume (limitação documentada:
# 698 SARIMAX × 12 folds é inviável em notebook; a prática recomendada para centenas de séries é o modelo global)
from statsmodels.tsa.statespace.sarimax import SARIMAX

top_series = list(painel.groupby(KEYS, observed=True)[ALVO].sum().nlargest(TOP_SARIMA).index)
series_top = set(top_series)
print(f"Amostra SARIMAX: {len(series_top)} séries mais representativas (por volume total de admissões)")

reg_sarima = []
for cutoff in FOLDS:
    treino = painel[painel["competencia_mov"] < cutoff]
    teste = painel[painel["competencia_mov"] == cutoff]
    denom = treino.groupby(KEYS, observed=True)[ALVO].agg(mase_denominador).rename("mase_den")
    teste_m = teste.merge(denom, left_on=KEYS, right_index=True, how="left")

    # Ajusta um SARIMAX por série da amostra e mapeia a previsão pela chave (UF, subgrupo)
    pred_map = {}
    for uf, sub in top_series:
        h = treino[(treino["sigla_uf"] == uf) & (treino["codigo_cbo_subgrupo"] == sub)]
        y = h[ALVO].values.astype(float)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fit = SARIMAX(np.log1p(y), order=(1, 0, 0), seasonal_order=(1, 0, 0, 12),
                              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            pred_map[(uf, sub)] = float(np.expm1(fit.forecast(1)[0]))
        except Exception:
            pred_map[(uf, sub)] = np.nan
    r = teste_m.copy()
    r["modelo"] = "sarimax_por_serie"
    r["previsto"] = [pred_map.get((u, s), np.nan) for u, s in zip(r["sigla_uf"], r["codigo_cbo_subgrupo"])]
    reg_sarima.append(r)

prev = pd.concat([prev] + reg_sarima, ignore_index=True)
print(f"SARIMAX concluído: {len(reg_sarima) * TOP_SARIMA} previsões adicionadas (amostra de {TOP_SARIMA} séries × {len(FOLDS)} folds)")

In [0]:
# Consolidação: métricas por modelo — geral (todas as séries) e na amostra SARIMAX (comparação justa)
def tabela_metricas(df, escopo):
    linhas = []
    for modelo, g in df.groupby("modelo"):
        gm = g.dropna(subset=["previsto"])
        err = (gm[ALVO] - gm["previsto"]).abs() / gm["mase_den"]
        linhas.append({
            "escopo": escopo, "modelo": modelo, "n": len(gm),
            "MAE": mae(gm[ALVO], gm["previsto"]), "RMSE": rmse(gm[ALVO], gm["previsto"]),
            "sMAPE": smape(gm[ALVO], gm["previsto"]),
            "MASE": float(err.replace([np.inf, -np.inf], np.nan).mean()),
        })
    return pd.DataFrame(linhas)

metricas_geral = tabela_metricas(prev, "TODAS_SERIES")
# SARIMAX só previu na amostra — seu escopo válido é AMOSTRA_SARIMAX (evita rótulo enganoso no geral)
metricas_geral = metricas_geral[metricas_geral["modelo"] != "sarimax_por_serie"]
prev_top = prev[prev[KEYS].apply(tuple, axis=1).isin(series_top)]
metricas_amostra = tabela_metricas(prev_top, "AMOSTRA_SARIMAX")
metricas = pd.concat([metricas_geral, metricas_amostra], ignore_index=True)

print("Métricas por modelo (escopo TODAS AS SÉRIES — MASE < 1 vence o naive sazonal in-sample):")
display(metricas_geral.sort_values("MASE").round(4))
print("\nNa amostra SARIMAX (top séries por volume — comparação justa com o SARIMAX por série):")
display(metricas_amostra.sort_values("MASE").round(4))

# Erro do melhor modelo ao longo dos folds (resíduos no tempo) + forecast vs real em séries representativas
melhor = metricas_geral.sort_values("MASE").iloc[0]["modelo"]
fig, ax = plt.subplots(figsize=(10, 3.5))
err_fold = prev[prev["modelo"] == melhor].dropna(subset=["previsto"]).groupby("competencia_mov").apply(
    lambda g: mae(g[ALVO], g["previsto"]), include_groups=False)
ax.plot(err_fold.index, err_fold.values, marker="o", ms=3, lw=1)
ax.set_title(f"MAE por fold — {melhor}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (uf, sub) in zip(axes, [("SP", "212"), ("SC", "231")]):
    s = painel[(painel["sigla_uf"] == uf) & (painel["codigo_cbo_subgrupo"] == sub)]
    ax.plot(s["competencia_mov"], s[ALVO], lw=1.2, label="real")
    for modelo in [melhor, "naive_sazonal_lag_12"]:
        p = prev[(prev["modelo"] == modelo) & (prev["sigla_uf"] == uf) & (prev["codigo_cbo_subgrupo"] == sub)]
        ax.plot(p["competencia_mov"], p["previsto"], marker="o", ms=2.5, lw=1, label=modelo)
    ax.set_title(f"{uf} × {sub} — holdout 2025", fontsize=10)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [0]:
# Gravação idempotente do backtest em gold — append por versao_backtest (sem DROP/overwrite)
linhas_gold = []

# Nível GERAL: métricas agregadas por modelo e escopo (consolidação acima)
for _, m in metricas.iterrows():
    linhas_gold.append({
        "nivel": "GERAL", "escopo": m["escopo"], "modelo": m["modelo"], "competencia_mov": None,
        "n": int(m["n"]), "mae": float(m["MAE"]), "rmse": float(m["RMSE"]),
        "smape": float(m["sMAPE"]), "mase": float(m["MASE"]),
        "versao_backtest": VERSAO_BACKTEST, "versao_features": VERSAO,
        "dt_processamento": DT_PROCESSAMENTO,
    })

# Nível FOLD: métricas por fold (escopo todas as séries; NaN de previsão excluído da métrica)
for modelo in prev["modelo"].unique():
    for cutoff in FOLDS:
        g = prev[(prev["modelo"] == modelo) & (prev["competencia_mov"] == cutoff)].dropna(subset=["previsto"])
        if g.empty:
            continue
        err = ((g[ALVO] - g["previsto"]).abs() / g["mase_den"]).replace([np.inf, -np.inf], np.nan)
        linhas_gold.append({
            "nivel": "FOLD", "escopo": "TODAS_SERIES", "modelo": modelo,
            "competencia_mov": cutoff, "n": len(g),
            "mae": mae(g[ALVO], g["previsto"]), "rmse": rmse(g[ALVO], g["previsto"]),
            "smape": smape(g[ALVO], g["previsto"]), "mase": float(err.mean()),
            "versao_backtest": VERSAO_BACKTEST, "versao_features": VERSAO,
            "dt_processamento": DT_PROCESSAMENTO,
        })

df_gold = pd.DataFrame(linhas_gold)
df_gold["competencia_mov"] = pd.to_datetime(df_gold["competencia_mov"])
df_gold["n"] = df_gold["n"].astype("int64")
df_gold["dt_processamento"] = pd.to_datetime(df_gold["dt_processamento"])

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABELA_BACKTEST} (
        nivel STRING COMMENT 'GERAL = agregado do holdout; FOLD = métrica de um cutoff',
        escopo STRING COMMENT 'TODAS_SERIES ou AMOSTRA_SARIMAX',
        modelo STRING COMMENT 'Modelo avaliado',
        competencia_mov DATE COMMENT 'Cutoff do fold (NULL no nível GERAL)',
        n BIGINT COMMENT 'Observações avaliadas',
        mae DOUBLE, rmse DOUBLE, smape DOUBLE, mase DOUBLE,
        versao_backtest STRING, versao_features STRING, dt_processamento TIMESTAMP
    )
""")

ja_existe = spark.sql(f"SELECT COUNT(*) FROM {TABELA_BACKTEST} WHERE versao_backtest = '{VERSAO_BACKTEST}'").collect()[0][0]
if ja_existe == 0:
    (spark.createDataFrame(df_gold).write
        .mode("append").saveAsTable(TABELA_BACKTEST))
    print(f"Backtest gravado (append, versão {VERSAO_BACKTEST}): {TABELA_BACKTEST} — {len(df_gold)} linhas")
else:
    print(f"Versão {VERSAO_BACKTEST} já gravada ({ja_existe} linhas) — append ignorado (idempotente).")
display(spark.table(TABELA_BACKTEST).where(F.col("versao_backtest") == VERSAO_BACKTEST)
    .where(F.col("nivel") == "GERAL").orderBy(F.col("mase")))

In [0]:
# MLflow: runs de backtest + modelo final refitado em toda a janela, logado como pyfunc de linhas futuras
import mlflow
from mlflow.models import infer_signature

nome_experimento = EXPERIMENTO
if not nome_experimento.startswith("/"):
    # nome relativo → caminho absoluto no workspace do usuário corrente
    usuario = spark.sql("SELECT current_user()").first()[0]
    nome_experimento = f"/Users/{usuario}/{nome_experimento}"
mlflow.set_experiment(nome_experimento)

# 1) Uma run por modelo × escopo com as métricas do holdout (SARIMAX só na amostra — evita rótulo enganoso)
for _, m in metricas.iterrows():
    if m["modelo"] == "sarimax_por_serie" and m["escopo"] != "AMOSTRA_SARIMAX":
        continue
    sufixo_escopo = "amostra" if m["escopo"] == "AMOSTRA_SARIMAX" else "todas"
    with mlflow.start_run(run_name=f"backtest_{m['modelo']}__{sufixo_escopo}"):
        mlflow.set_tags({"modelo": m["modelo"], "escopo": m["escopo"], "versao_features": VERSAO, "versao_backtest": VERSAO_BACKTEST})
        mlflow.log_params({"janela_inicio": JANELA_INICIO, "janela_fim": JANELA_FIM, "folds": len(FOLDS),
                           "horizonte": HORIZONTE, "n_series": n_series, "top_sarima": TOP_SARIMA})
        mlflow.log_metrics({"mae": float(m["MAE"]), "rmse": float(m["RMSE"]),
                            "smape": float(m["sMAPE"]), "mase": float(m["MASE"]), "n_obs": float(m["n"])})

# 2) Refit final do LightGBM em TODA a janela (treino + holdout) — previsões começam em 2026-01
PARAMS_LGBM = dict(n_estimators=400, learning_rate=0.05, num_leaves=31,
                    min_child_samples=20, random_state=42, verbosity=-1)
fit_final = lgb.LGBMRegressor(**PARAMS_LGBM).fit(
    painel[FEAT_LGBM], painel[ALVO], categorical_feature=["sigla_uf", "codigo_cbo_subgrupo"])

# Histórico por série para o wrapper: 12-13 últimos meses com y, desligamentos e salário
# (salário do mês m = salario_mediano_lag_1 da linha m+1 — por isso carregamos 2026-01 no início)
full = pd.concat([painel, hist_extra], ignore_index=True).sort_values(KEYS + ["competencia_mov"])
full["sal_hist"] = full.groupby(KEYS, observed=True)["salario_mediano_lag_1"].shift(-1)
hist = full[full["competencia_mov"] >= "2025-01-01"][KEYS + ["competencia_mov", ALVO, "desligamentos", "sal_hist"]]
hist = hist.rename(columns={ALVO: "y", "desligamentos": "desl", "sal_hist": "sal"})

LAGS_Y = list(range(1, 13))  # 1..12 (lag >= h=1; ma_12 exige todos)
CAT_MAPS = {c: painel[c].cat.categories for c in ["sigla_uf", "codigo_cbo_subgrupo"]}
# O histórico veio do concat com hist_extra (chaves em object) — reintroduz as categorias do treino
for c in CAT_MAPS:
    hist[c] = pd.Categorical(hist[c], categories=CAT_MAPS[c])

class GlobalForecaster(mlflow.pyfunc.PythonModel):
    """Recebe linhas futuras (sigla_uf, codigo_cbo_subgrupo, competencia_mov) e devolve a previsão
    de admissões (h=1..12). Os lags/rollings são recomputados do histórico — quem chama não
    reconstrói features."""
    def __init__(self, regressor, hist, cat_maps, features):
        self.regressor, self.hist, self.cat_maps, self.features = regressor, hist, cat_maps, features
    def predict(self, context, model_input, params=None):
        fut = model_input.copy()
        fut["competencia_mov"] = pd.to_datetime(fut["competencia_mov"])
        for c, cats in self.cat_maps.items():
            fut[c] = pd.Categorical(fut[c], categories=cats)
        h = self.hist
        for lag in LAGS_Y:
            lagged = h[KEYS + ["competencia_mov", "y"]].rename(columns={"y": f"admissoes_lag_{lag}"})
            lagged["competencia_mov"] = lagged["competencia_mov"] + pd.DateOffset(months=lag)
            fut = fut.merge(lagged, on=KEYS + ["competencia_mov"], how="left")
        for col, feat in [("desl", "desligamentos_lag_1"), ("sal", "salario_mediano_lag_1")]:
            lagged = h[KEYS + ["competencia_mov", col]].rename(columns={col: feat})
            lagged["competencia_mov"] = lagged["competencia_mov"] + pd.DateOffset(months=1)
            fut = fut.merge(lagged, on=KEYS + ["competencia_mov"], how="left")
        fut["admissoes_ma_3"] = fut[[f"admissoes_lag_{k}" for k in (1, 2, 3)]].mean(axis=1)
        fut["admissoes_ma_12"] = fut[[f"admissoes_lag_{k}" for k in LAGS_Y]].mean(axis=1)
        fut["saldo_lag_1"] = fut["admissoes_lag_1"] - fut["desligamentos_lag_1"]
        fut["mes"] = fut["competencia_mov"].dt.month
        out = fut.copy()
        # merges podem rebaixar as chaves a object — reintroduz as categorias EXATAS do treino
        for c, cats in self.cat_maps.items():
            out[c] = pd.Categorical(out[c], categories=cats)
        out["forecast"] = self.regressor.predict(out[self.features])
        res = out[[*KEYS, "competencia_mov", "forecast"]].copy()
        # saída com chaves como string comum (esquema limpo para a assinatura MLflow)
        for c in self.cat_maps:
            res[c] = res[c].astype(str)
        return res

modelo_final = GlobalForecaster(fit_final, hist, CAT_MAPS, FEAT_LGBM)

# Linhas futuras de verificação: TODAS as séries × 2026-01 (mês seguinte ao fim da janela)
# Chaves como string comum — a assinatura/entrada do MLflow espera tipos simples; o wrapper
# reintroduz as categorias do treino internamente em predict().
future = painel[KEYS].drop_duplicates().assign(competencia_mov=pd.Timestamp("2026-01-01")).copy()
future[KEYS[0]] = future[KEYS[0]].astype(str)
future[KEYS[1]] = future[KEYS[1]].astype(str)
forecast = modelo_final.predict(None, future)
assert len(forecast) == len(future) and forecast["forecast"].notna().all(), "wrapper não previu todas as linhas futuras"

with mlflow.start_run(run_name="final_lightgbm_h1"):
    mlflow.set_tags({"modelo": "lightgbm_global", "versao_features": VERSAO, "versao_backtest": VERSAO_BACKTEST,
                     "estagio": "refit_final"})
    mlflow.log_params({**PARAMS_LGBM, "janela_treino": f"{JANELA_INICIO}..{JANELA_FIM}",
                       "n_series": n_series, "n_treino_final": len(painel)})
    ml_hold = metricas_geral[metricas_geral["modelo"] == "lightgbm_global"].iloc[0]
    mlflow.log_metrics({"mae": float(ml_hold["MAE"]), "rmse": float(ml_hold["RMSE"]),
                        "smape": float(ml_hold["sMAPE"]), "mase": float(ml_hold["MASE"])})
    model_info = mlflow.pyfunc.log_model(
        python_model=modelo_final, name="modelo",
        signature=infer_signature(future, forecast), input_example=future.head(5),
        extra_pip_requirements=[f"lightgbm=={lgb.__version__}"],
    )

# Verificação a partir do artefato logado (não de objetos em memória)
carregado = mlflow.pyfunc.load_model(model_info.model_uri)
prev_verif = carregado.predict(future.head(100))
assert len(prev_verif) == 100 and prev_verif["forecast"].notna().all(), "modelo logado falhou na previsão de linhas futuras"
print(f"Modelo final logado: {model_info.model_uri}")
print(f"Verificação do artefato: 100 linhas futuras previstas sem nulos. Exemplo (2026-01):")
display(prev_verif.sort_values("forecast", ascending=False).head(10))

In [0]:
%pip install uv
# Registro do modelo no Unity Catalog — REUTILIZA o artefato logado (sem retreinar/re-logar).
# Fluxo: checagem isolada de servimento (ambiente limpo via uv) → registro databricks-uc.
import mlflow
from mlflow import MlflowClient

MODEL_URI = "models:/m-c3989b6679674aabb49e4cf09fbf4501"  # artefato da run final_lightgbm_h1 (experimento demandA_admissoes_h1)
NOME_REGISTRADO = "workspace.gold.demanda_admissoes_lgbm"  # catalog.schema.model do projeto

# 1) Checagem isolada: assinatura + input example + predição com instalação limpa das dependências
#    (um container de serving instala os pip_requirements do zero — só isso prova servibilidade)
pyfunc_model = mlflow.pyfunc.load_model(MODEL_URI)
assert pyfunc_model.metadata.signature is not None, "Artefato sem assinatura — corrigir antes de registrar."
exemplo = pyfunc_model.input_example
assert exemplo is not None, "Artefato sem input example — corrigir antes de registrar."

import tempfile
caminho_saida = tempfile.NamedTemporaryFile(suffix=".json", delete=False).name

resultado_check = mlflow.models.predict(
    model_uri=MODEL_URI, input_data=exemplo, env_manager="uv",
    output_path=caminho_saida,  # nesta versão do MLflow, predict NÃO retorna o resultado — escreve no arquivo
)
import json, math
with open(caminho_saida) as f:
    saida = json.load(f)
# Formatos possíveis do JSON de saída (lista de registros em "predictions", split-orient ou lista simples)
if isinstance(saida, dict) and "predictions" in saida:
    valores = [r["forecast"] for r in saida["predictions"]]
elif isinstance(saida, dict) and "data" in saida:
    colunas = saida.get("columns", [])
    idx_f = colunas.index("forecast") if "forecast" in colunas else len(colunas) - 1
    valores = [linha[idx_f] for linha in saida["data"]]
elif isinstance(saida, list) and saida and isinstance(saida[0], list):
    valores = [linha[-1] for linha in saida]
else:
    valores = saida
assert valores is not None and len(valores) == len(exemplo), f"Checagem isolada falhou: {saida!r}"
assert all(v is not None and not (isinstance(v, float) and math.isnan(v)) for v in valores), f"Previsões nulas: {valores!r}"
print(f"Checagem isolada de servimento OK (uv, instalação limpa): {len(valores)} previsões — {valores}")

# 2) Registro idempotente: se este MESMO artefato já tem versão no UC, não cria outra
mlflow.set_registry_uri("databricks-uc")
reg_client = MlflowClient(registry_uri="databricks-uc")
ja_registradas = []
try:
    ja_registradas = [v for v in reg_client.search_model_versions(f"name='{NOME_REGISTRADO}'")
                      if v.source == MODEL_URI]
except Exception:
    pass  # modelo ainda não existe no UC — segue para o registro

if ja_registradas:
    versao_modelo = ja_registradas[0].version
    print(f"Artefato já registrado: {NOME_REGISTRADO} v{versao_modelo} — nenhuma nova versão criada (idempotente)")
else:
    registrado = mlflow.register_model(model_uri=MODEL_URI, name=NOME_REGISTRADO, await_registration_for=300)
    versao_modelo = registrado.version
    print(f"Registrado no Unity Catalog: {NOME_REGISTRADO} versão {versao_modelo}")

print(f"URI para serving: models:/{NOME_REGISTRADO}/{versao_modelo}")

In [0]:
# Serving: cria (ou atualiza) o endpoint do modelo registrado — idempotente, não recria
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput, ServingModelWorkloadType,
)

NOME_ENDPOINT = "demanda_admissoes_lgbm"
MODELO_SERVING = "workspace.gold.demanda_admissoes_lgbm"
VERSAO_SERVING = "1"  # pinada na versão exata — upgrade/rollback = trocar aqui + update_config

w = WorkspaceClient()
entidade = ServedEntityInput(
    entity_name=MODELO_SERVING,
    entity_version=VERSAO_SERVING,
    workload_type=ServingModelWorkloadType.CPU,  # menor tipo de CPU suportado — modelo leve (LGBM + histórico embutido)
    workload_size="Small",
    scale_to_zero_enabled=True,  # sem custo ocioso; a 1ª chamada após inatividade tem cold start
)

try:
    atual = w.serving_endpoints.get(NOME_ENDPOINT)
    served = atual.config.served_entities[0]
    if served.entity_name == MODELO_SERVING and served.entity_version == VERSAO_SERVING:
        print(f"Endpoint '{NOME_ENDPOINT}' já existe servindo {MODELO_SERVING} v{VERSAO_SERVING} — nada a fazer.")
    else:
        w.serving_endpoints.update_config(name=NOME_ENDPOINT, served_entities=[entidade])
        print(f"Endpoint '{NOME_ENDPOINT}' atualizado para {MODELO_SERVING} v{VERSAO_SERVING}.")
except Exception:
    w.serving_endpoints.create(
        name=NOME_ENDPOINT,
        config=EndpointCoreConfigInput(name=NOME_ENDPOINT, served_entities=[entidade]),
    )
    print(f"Endpoint '{NOME_ENDPOINT}' criado servindo {MODELO_SERVING} v{VERSAO_SERVING} (CPU Small, scale-to-zero).")
    print("Aguardar readiness na célula seguinte.")


In [0]:
# Serving: polling de readiness — o build do container costuma levar ~15 min
# (reexecute esta célula se estourar as tentativas — o build continua no servidor)
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady

NOME_ENDPOINT = "demanda_admissoes_lgbm"
w = WorkspaceClient()

pronto = False
for tentativa in range(1, 46):  # até ~15 min (20s × 45)
    ep = w.serving_endpoints.get(NOME_ENDPOINT)
    if ep.state.ready == EndpointStateReady.READY:
        print(f"Endpoint '{NOME_ENDPOINT}' READY.")
        print(f"Scoring URL: {w.config.host}/serving-endpoints/{NOME_ENDPOINT}/invocations")
        pronto = True
        break
    if ep.state.config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
        raise RuntimeError(f"Build FALHOU: {ep.state.config_update}")
    print(f"[{tentativa:02d}/45] {ep.state.config_update} — aguardando build...")
    time.sleep(20)

if not pronto:
    print("Build ainda em andamento após ~15 min — reexecute esta célula para continuar aguardando.")


## Observações e limitações

### Resultados do backtest (v0.1.0 — 12 folds expansivos, 2025-01 a 2025-12, h=1)

| Modelo | MAE | RMSE | sMAPE | MASE |
| --- | --- | --- | --- | --- |
| lightgbm_global | **21,5** | **72,2** | 0,493 | 1,357 |
| naive_sazonal_lag_12 | 24,4 | 75,9 | 0,434 | 1,113 |
| ols_pooled | 36,8 | 155,8 | **0,395** | **1,107** |
| naive_lag_1 | 43,5 | 194,6 | 0,499 | 1,513 |
| sarimax_por_serie (amostra top-20) | 214,9* | 392,9* | 0,095* | 0,931* |

\* métricas da amostra top-20 — na mesma amostra o LightGBM faz MASE 0,80 e MAE 192,6, vencendo a naive sazonal (0,86) e o SARIMAX (0,93).

* **Leitura honesta** — o LightGBM global é o melhor em MAE/RMSE no painel inteiro e vence a baseline sazonal onde está o volume (séries grandes), mas o MASE agregado (1,36) perde para OLS/naive (≈1,11): o MASE pesa todas as séries igualmente e o LightGBM superestima séries pequenas. Evoluções naturais: treinar em escala log, segmentar por volume, ou calibrar por faixa de tamanho de série.
* **Modelo final** — LightGBM refitado em toda a janela (2022-01..2025-12), logado no MLflow como pyfunc que recebe linhas futuras (`models:/m-c3989b6679674aabb49e4cf09fbf4501`), com assinatura e verificação por artefato (previsões de 2026-01 sem nulos). **Registrado no Unity Catalog como `workspace.gold.demanda_admissoes_lgbm` versão 1** (`models:/workspace.gold.demanda_admissoes_lgbm/1`, status READY) após checagem isolada de servimento em ambiente limpo (uv) — o mesmo artefato, sem retreino. O experimento também contém runs duplicadas das tentativas de execução da célula MLflow — podem ser limpas no UI do experimento.

### Decisões e limitações

* **Janela expansiva** com 12 folds mensais, horizonte 1, sem embaralhamento; naive sazonal (lag 12) como baseline obrigatória (MASE = 1 por construção da denominação).
* **SARIMAX por série** roda nas top-20 séries por volume como referência clássica — modelo global é a prática recomendada para centenas de séries (698 aqui).
* **OLS em log1p** — previsão via `expm1` sem fator de correção de Jensen (viés pequeno, documentado). LightGBM treinado no alvo em escala original.
* **Sem features INEP no modelo final** — screening do 035 as descartou (p>0,25) e elas zeram a partir de 2026; previsões de 2026 seguem possíveis com features CAGED defasadas + calendário.
* **Salário mediano nulo** (mês anterior sem admissões): LightGBM trata NaN nativamente; OLS exclui a linha (7.492 das 8.376 previsões válidas).
* **Idempotência** — backtest gravado por append em `workspace.gold.ml_backtest_admissoes` por `versao_backtest`; re-execução com a mesma versão não duplica.
* **Auditoria de vazamento como gate** — antes do treino, uma célula permanente recomputa independentemente (SQL com funções de janela) todos os lags e médias móveis da wide table e compara com as colunas gravadas; também confere que as médias móveis excluem o mês corrente e que a feature INEP é constante dentro do ano. Qualquer violação aborta a execução antes de treinar.
* **Evoluções** — hiperparâmetros fixos (busca sistemática fica como evolução); Prophet opcional; fill-forward anual das features INEP quando o Censo 2025 chegar; recursive forecasting para horizontes h>1.

**Model Serving** — endpoint `demanda_admissoes_lgbm` (CPU Small, scale-to-zero) criado pela célula idempotente após o registro; consultas via REST aceitam linhas futuras (UF, subgrupo, competência) no mesmo contrato do wrapper pyfunc.